<a href="https://colab.research.google.com/github/Shreya-1910/ArtificalMethodsCW/blob/main/pso-ga.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pyswarms --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pickle
import json
import time
import os
import gc
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight
import pyswarms as ps

print("All libraries loaded!")


2026-04-18 12:01:32,734 - numexpr.utils - INFO - NumExpr defaulting to 2 threads.


All libraries loaded!


In [3]:

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import os

# Correct path from diagnostic
DATA_PATH = '/content/drive/MyDrive/AIMcw/CICIDS2017/preprocess/'

print("=" * 60)
print("LOADING DATA FROM:", DATA_PATH)
print("=" * 60)

# Load data
X_train = np.load(DATA_PATH + 'x_train.npy')
X_val = np.load(DATA_PATH + 'x_val.npy')
X_test = np.load(DATA_PATH + 'x_test.npy')
y_train = np.load(DATA_PATH + 'y_train.npy')
y_val = np.load(DATA_PATH + 'y_val.npy')
y_test = np.load(DATA_PATH + 'y_test.npy')

print(f" Data loaded successfully!")
print(f"   Training:   {X_train.shape[0]:,} × {X_train.shape[1]}")
print(f"   Validation: {X_val.shape[0]:,} × {X_val.shape[1]}")
print(f"   Test:       {X_test.shape[0]:,} × {X_test.shape[1]}")

# Check if feature names exist
feature_names_path = DATA_PATH + 'feature_names.txt'
if os.path.exists(feature_names_path):
    with open(feature_names_path, 'r') as f:
        FEATURE_NAMES = [line.strip() for line in f.readlines()]
    print(f"   Features:   {len(FEATURE_NAMES)} named features")
else:
    print("   Warning: feature_names.txt not found")
    FEATURE_NAMES = [f"Feature_{i}" for i in range(X_train.shape[1])]

# Check class distribution
print(f"\n Class Distribution:")
print(f"   Training:   Benign={(y_train==0).sum():,}, Attack={(y_train==1).sum():,}")
print(f"   Validation: Benign={(y_val==0).sum():,}, Attack={(y_val==1).sum():,}")
print(f"   Test:       Benign={(y_test==0).sum():,}, Attack={(y_test==1).sum():,}")

N_FEATURES = X_train.shape[1]
print(f"\n Optimization target: {N_FEATURES} features to select from")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOADING DATA FROM: /content/drive/MyDrive/AIMcw/CICIDS2017/preprocess/
 Data loaded successfully!
   Training:   1,765,652 × 70
   Validation: 252,237 × 70
   Test:       504,473 × 70
   Features:   70 named features

 Class Distribution:
   Training:   Benign=1,467,538, Attack=298,114
   Validation: Benign=209,649, Attack=42,588
   Test:       Benign=419,297, Attack=85,176

 Optimization target: 70 features to select from


In [4]:

print("=" * 60)
print("EXTERNAL BENCHMARK BASELINE")
print("=" * 60)

# Baseline performance from independent experiment
# Random Forest trained on 100% training data using all 70 features
BASELINE_PERFORMANCE = {
    "accuracy": 0.9977,
    "precision": 0.9977,
    "recall": 0.9977,
    "f1_score": 0.9977,
    "roc_auc": 0.9998,
    "n_features": 70,
    "training_samples": 1765652,
    "training_time_sec": 144.5,
    "model_architecture": "Random Forest (n_estimators=100, max_depth=20)"
}

print(f"""
Benchmark Baseline Performance (External Reference):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Metric         | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy:      {BASELINE_PERFORMANCE['accuracy']:.4f}
  Precision:     {BASELINE_PERFORMANCE['precision']:.4f}
  Recall:        {BASELINE_PERFORMANCE['recall']:.4f}
  F1-Score:      {BASELINE_PERFORMANCE['f1_score']:.4f}
  ROC-AUC:       {BASELINE_PERFORMANCE['roc_auc']:.4f}
  Features used: {BASELINE_PERFORMANCE['n_features']}
  Training data: {BASELINE_PERFORMANCE['training_samples']:,} samples
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OPTIMIZATION TARGET: Achieve F1-Score > {BASELINE_PERFORMANCE['f1_score']:.4f}
""")

EXTERNAL BENCHMARK BASELINE

Benchmark Baseline Performance (External Reference):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Metric         | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy:      0.9977
  Precision:     0.9977
  Recall:        0.9977
  F1-Score:      0.9977
  ROC-AUC:       0.9998
  Features used: 70
  Training data: 1,765,652 samples
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OPTIMIZATION TARGET: Achieve F1-Score > 0.9977



In [5]:


from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

print("=" * 60)
print("CREATING OPTIMIZATION SUBSET (5% OF TRAINING DATA)")
print("=" * 60)

# Parameters
OPTIMIZATION_PERCENTAGE = 0.05
RANDOM_STATE = 42

# Calculate sample size
optimization_sample_size = int(OPTIMIZATION_PERCENTAGE * X_train.shape[0])

print(f"""
Optimization Strategy:
  Full training data: {X_train.shape[0]:,} samples
  Optimization subset: {optimization_sample_size:,} samples ({OPTIMIZATION_PERCENTAGE*100:.0f}%)
  Rationale: PSO-GA requires 1,500+ Random Forest trainings
             Running on full data would take >100 hours
             Optimization on 5% completes in ~4 hours
""")

# Stratified sampling to preserve class distribution
stratified_sampler = StratifiedShuffleSplit(
    n_splits=1,
    test_size=optimization_sample_size,
    random_state=RANDOM_STATE
)
_, optimization_indices = next(stratified_sampler.split(X_train, y_train))

# Extract optimization subset
X_optimization = X_train[optimization_indices]
y_optimization = y_train[optimization_indices]

print(f"\nOptimization subset composition:")
print(f"  Total samples: {X_optimization.shape[0]:,}")
print(f"  Benign: {(y_optimization==0).sum():,} ({((y_optimization==0).sum()/len(y_optimization))*100:.1f}%)")
print(f"  Attack: {(y_optimization==1).sum():,} ({((y_optimization==1).sum()/len(y_optimization))*100:.1f}%)")

# Split optimization subset into training and validation (80/20)
X_opt_train, X_opt_val, y_opt_train, y_opt_val = train_test_split(
    X_optimization, y_optimization,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_optimization
)

print(f"\nOptimization internal split:")
print(f"  Training (for RF inside PSO/GA): {X_opt_train.shape[0]:,} samples")
print(f"  Validation (for fitness evaluation): {X_opt_val.shape[0]:,} samples")

print(f"\nData Usage Summary:")

print(f"  │ PSO-GA OPTIMIZATION: {X_optimization.shape[0]:,} samples ({OPTIMIZATION_PERCENTAGE*100:.0f}%) │")
print(f"  │ FINAL MODEL TRAINING: {X_train.shape[0]:,} samples (100%)        │")


CREATING OPTIMIZATION SUBSET (5% OF TRAINING DATA)

Optimization Strategy:
  Full training data: 1,765,652 samples
  Optimization subset: 88,282 samples (5%)
  Rationale: PSO-GA requires 1,500+ Random Forest trainings
             Running on full data would take >100 hours
             Optimization on 5% completes in ~4 hours


Optimization subset composition:
  Total samples: 88,282
  Benign: 73,376 (83.1%)
  Attack: 14,906 (16.9%)

Optimization internal split:
  Training (for RF inside PSO/GA): 70,625 samples
  Validation (for fitness evaluation): 17,657 samples

Data Usage Summary:
  │ PSO-GA OPTIMIZATION: 88,282 samples (5%) │
  │ FINAL MODEL TRAINING: 1,765,652 samples (100%)        │


In [6]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

print("=" * 60)
print("DEFINING FITNESS FUNCTIONS")
print("=" * 60)

# Target F1 from external baseline
TARGET_F1 = BASELINE_PERFORMANCE['f1_score']

def pso_fitness_function(particles):
    """
    Fitness function for Particle Swarm Optimization.
    Minimizes negative F1-score with feature penalty.

    Parameters:
    particles: numpy array of particle positions (70 features + 4 hyperparameters)

    Returns:
    numpy array of fitness scores (lower is better)
    """
    scores = []

    for particle in particles:
        # Extract feature selection mask (first 70 dimensions)
        feature_mask = particle[:N_FEATURES] > 0.5

        # Penalize empty feature selection
        if feature_mask.sum() == 0:
            scores.append(1.0)
            continue

        # Extract hyperparameters (last 4 dimensions)
        n_estimators = int(np.clip(particle[N_FEATURES] * 150 + 50, 50, 200))
        max_depth = int(np.clip(particle[N_FEATURES + 1] * 30 + 5, 5, 35))
        min_samples_split = int(np.clip(particle[N_FEATURES + 2] * 18 + 2, 2, 20))
        min_samples_leaf = int(np.clip(particle[N_FEATURES + 3] * 9 + 1, 1, 10))

        # Lightweight Random Forest for efficient optimization
        rf = RandomForestClassifier(
            n_estimators=min(30, n_estimators),
            max_depth=min(15, max_depth),
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        )

        # Train and evaluate
        rf.fit(X_opt_train[:, feature_mask], y_opt_train)
        y_pred = rf.predict(X_opt_val[:, feature_mask])
        f1 = f1_score(y_opt_val, y_pred, average='weighted', zero_division=0)

        # Regularization terms
        feature_penalty = 0.0005 * feature_mask.sum()  # Encourage fewer features
        benchmark_bonus = max(0, (f1 - TARGET_F1) * 10) if f1 > TARGET_F1 else 0

        # Fitness (minimize negative F1)
        scores.append(-f1 + feature_penalty - benchmark_bonus)

    return np.array(scores)


def ga_fitness_function(chromosome):
    """
    Fitness function for Genetic Algorithm.
    Maximizes F1-score with feature penalty.

    Parameters:
    chromosome: binary array (70 feature bits + 32 hyperparameter bits)

    Returns:
    float: fitness score (higher is better)
    """
    # Extract feature mask (first 70 bits)
    feature_mask = chromosome[:N_FEATURES] == 1

    if feature_mask.sum() == 0:
        return 0.0

    # Decode hyperparameters from binary representation (next 32 bits)
    if len(chromosome) > N_FEATURES:
        # n_estimators: 8 bits (range 50-200)
        n_est_bits = chromosome[N_FEATURES:N_FEATURES+8]
        n_estimators = 50 + int(''.join(map(str, n_est_bits.astype(int))), 2) % 151

        # max_depth: 8 bits (range 5-35)
        max_depth_bits = chromosome[N_FEATURES+8:N_FEATURES+16]
        max_depth = 5 + int(''.join(map(str, max_depth_bits.astype(int))), 2) % 31

        # min_samples_split: 8 bits (range 2-20)
        min_split_bits = chromosome[N_FEATURES+16:N_FEATURES+24]
        min_samples_split = 2 + int(''.join(map(str, min_split_bits.astype(int))), 2) % 19

        # min_samples_leaf: 8 bits (range 1-10)
        min_leaf_bits = chromosome[N_FEATURES+24:N_FEATURES+32]
        min_samples_leaf = 1 + int(''.join(map(str, min_leaf_bits.astype(int))), 2) % 10
    else:
        # Default hyperparameters if not encoded
        n_estimators, max_depth, min_samples_split, min_samples_leaf = 50, 15, 5, 2

    # Lightweight Random Forest
    rf = RandomForestClassifier(
        n_estimators=min(30, n_estimators),
        max_depth=min(15, max_depth),
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )

    # Train and evaluate
    rf.fit(X_opt_train[:, feature_mask], y_opt_train)
    y_pred = rf.predict(X_opt_val[:, feature_mask])
    f1 = f1_score(y_opt_val, y_pred, average='weighted', zero_division=0)

    # Feature penalty for parsimony
    feature_penalty = 0.0003 * feature_mask.sum()

    return f1 - feature_penalty

print(f" Fitness functions defined successfully")
print(f"   Target F1 to beat: {TARGET_F1:.4f}")
print(f"   Optimization training data: {X_opt_train.shape[0]:,} samples")
print(f"   Optimization validation data: {X_opt_val.shape[0]:,} samples")
print(f"   Search space dimensions: {N_FEATURES + 4} (PSO) / {N_FEATURES + 32} (GA)")

DEFINING FITNESS FUNCTIONS
 Fitness functions defined successfully
   Target F1 to beat: 0.9977
   Optimization training data: 70,625 samples
   Optimization validation data: 17,657 samples
   Search space dimensions: 74 (PSO) / 102 (GA)


In [9]:


import pyswarms as ps
import time

print("=" * 60)
print("PHASE 1: PARTICLE SWARM OPTIMIZATION")
print("=" * 60)

# PSO Configuration
N_PARTICLES = 25
N_ITERATIONS = 20
N_DIMENSIONS = N_FEATURES + 4  # 70 features + 4 hyperparameters

PSO_PARAMETERS = {
    'c1': 0.5,  # Cognitive component (personal best influence)
    'c2': 0.3,  # Social component (global best influence)
    'w': 0.9    # Inertia weight (exploration vs exploitation)
}

BOUNDS = (np.zeros(N_DIMENSIONS), np.ones(N_DIMENSIONS))

print(f"""
PSO Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Parameter           | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Particles:          | {N_PARTICLES}
  Iterations:         | {N_ITERATIONS} (reduced from 25 to 20)
  Dimensions:         | {N_DIMENSIONS} (70 features + 4 hyperparameters)
  Cognitive (c1):     | {PSO_PARAMETERS['c1']}
  Social (c2):        | {PSO_PARAMETERS['c2']}
  Inertia (w):        | {PSO_PARAMETERS['w']}
  Training data:      | {X_opt_train.shape[0]:,} samples
  Validation data:    | {X_opt_val.shape[0]:,} samples
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Estimated runtime: 2.5-3 hours (saves ~1 hour)
""")

# Initialize PSO optimizer
pso_optimizer = ps.single.GlobalBestPSO(
    n_particles=N_PARTICLES,
    dimensions=N_DIMENSIONS,
    options=PSO_PARAMETERS,
    bounds=BOUNDS
)

# Run optimization
print(" Starting PSO optimization...")
print("   (Progress bar will show below)")
print()

pso_start_time = time.time()
best_cost, best_position = pso_optimizer.optimize(
    pso_fitness_function,
    iters=N_ITERATIONS,
    verbose=True
)
pso_duration = (time.time() - pso_start_time) / 60

print(f"\nPSO optimization completed in {pso_duration:.1f} minutes")
print(f"   Best cost (fitness): {best_cost:.6f}")

# Extract optimal solution from PSO
pso_feature_mask = best_position[:N_FEATURES] > 0.5
pso_n_features = int(pso_feature_mask.sum())

pso_n_estimators = int(np.clip(best_position[N_FEATURES] * 150 + 50, 50, 200))
pso_max_depth = int(np.clip(best_position[N_FEATURES + 1] * 30 + 5, 5, 35))
pso_min_samples_split = int(np.clip(best_position[N_FEATURES + 2] * 18 + 2, 2, 20))
pso_min_samples_leaf = int(np.clip(best_position[N_FEATURES + 3] * 9 + 1, 1, 10))

print(f"\n Best Solution Found by PSO:")
print(f"  Selected features: {pso_n_features}/{N_FEATURES} ({pso_n_features/N_FEATURES*100:.1f}%)")
print(f"  Hyperparameters:")
print(f"    n_estimators:      {pso_n_estimators}")
print(f"    max_depth:         {pso_max_depth}")
print(f"    min_samples_split: {pso_min_samples_split}")
print(f"    min_samples_leaf:  {pso_min_samples_leaf}")

2026-04-18 12:55:45,026 - pyswarms.single.global_best - INFO - Optimize for 20 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}


PHASE 1: PARTICLE SWARM OPTIMIZATION

PSO Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Parameter           | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Particles:          | 25
  Iterations:         | 20 (reduced from 25 to 20)
  Dimensions:         | 74 (70 features + 4 hyperparameters)
  Cognitive (c1):     | 0.5
  Social (c2):        | 0.3
  Inertia (w):        | 0.9
  Training data:      | 70,625 samples
  Validation data:    | 17,657 samples
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Estimated runtime: 2.5-3 hours (saves ~1 hour)

 Starting PSO optimization...
   (Progress bar will show below)



pyswarms.single.global_best: 100%|██████████|20/20, best_cost=-0.996
2026-04-18 13:25:05,800 - pyswarms.single.global_best - INFO - Optimization finished | best cost: -0.9962943874950446, best pos: [0.94253177 0.43740493 0.54151307 0.27777076 0.98180049 0.04654148
 0.44076863 0.06912491 0.8205296  0.76152219 0.68966599 0.12753636
 0.38488266 0.34050249 0.35353977 0.43225441 0.276509   0.01643502
 0.0325365  0.50114373 0.78791189 0.37752164 0.90149924 0.39707327
 0.72212676 0.83326935 0.57355873 0.05656974 0.9642212  0.67134062
 0.44323954 0.31344021 0.57185655 0.60667948 0.87942432 0.47048001
 0.43838227 0.49202743 0.73697697 0.42997545 0.10084908 0.26076135
 0.16305441 0.21506893 0.82214498 0.39127717 0.66444898 0.28806064
 0.18840727 0.28398689 0.9594419  0.49695911 0.16223235 0.20761428
 0.84480257 0.18594013 0.32106092 0.40983861 0.59664162 0.10334985
 0.04622834 0.37773478 0.58425009 0.48394603 0.80913122 0.45352657
 0.13876786 0.23752565 0.5607588  0.42816876 0.69594549 0.8605619


PSO optimization completed in 29.3 minutes
   Best cost (fitness): -0.996294

 Best Solution Found by PSO:
  Selected features: 26/70 (37.1%)
  Hyperparameters:
    n_estimators:      154
    max_depth:         30
    min_samples_split: 6
    min_samples_leaf:  2


In [10]:


print("=" * 60)
print("PHASE 2: GENETIC ALGORITHM REFINEMENT")
print("=" * 60)

# GA Configuration
POPULATION_SIZE = 30
N_GENERATIONS = 15
MUTATION_RATE = 0.05
ELITE_SIZE = 3
TOURNAMENT_SIZE = 3
CROSSOVER_RATE = 0.9

print(f"""
GA Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Parameter           | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Population size:    | {POPULATION_SIZE}
  Generations:        | {N_GENERATIONS}
  Mutation rate:      | {MUTATION_RATE}
  Elite size:         | {ELITE_SIZE}
  Tournament size:    | {TOURNAMENT_SIZE}
  Crossover rate:     | {CROSSOVER_RATE}
  Seed:               | PSO optimal solution injected
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Estimated runtime: 1-2 hours
""")

# Initialize population with PSO solution as seed
population = np.random.randint(0, 2, (POPULATION_SIZE, N_FEATURES + 32))

# Encode PSO solution as seed chromosome
pso_seed = np.zeros(N_FEATURES + 32)
pso_seed[:N_FEATURES] = best_position[:N_FEATURES] > 0.5
pso_seed[N_FEATURES:N_FEATURES+8] = [int(b) for b in format(pso_n_estimators - 50, '08b')]
pso_seed[N_FEATURES+8:N_FEATURES+16] = [int(b) for b in format(pso_max_depth - 5, '08b')]
pso_seed[N_FEATURES+16:N_FEATURES+24] = [int(b) for b in format(pso_min_samples_split - 2, '08b')]
pso_seed[N_FEATURES+24:N_FEATURES+32] = [int(b) for b in format(pso_min_samples_leaf - 1, '08b')]

population[0] = pso_seed  # Inject PSO solution as first chromosome

# Genetic operators
def tournament_selection(pop, scores, k=TOURNAMENT_SIZE):
    """Select parent using tournament selection."""
    indices = np.random.choice(len(pop), k, replace=False)
    return pop[indices[np.argmax(scores[indices])]].copy()

def single_point_crossover(parent1, parent2, rate=CROSSOVER_RATE):
    """Perform single-point crossover."""
    if np.random.rand() < rate:
        crossover_point = np.random.randint(1, len(parent1))
        child1 = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])
        child2 = np.concatenate([parent2[:crossover_point], parent1[crossover_point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()

def bit_flip_mutation(chromosome, rate=MUTATION_RATE):
    """Apply bit-flip mutation."""
    flip_mask = np.random.rand(len(chromosome)) < rate
    chromosome[flip_mask] = 1 - chromosome[flip_mask]
    return chromosome

# Run GA
print(" Starting GA optimization...")
print()

ga_start_time = time.time()
best_fitness_history = []
avg_fitness_history = []

for generation in range(N_GENERATIONS):
    # Evaluate fitness for entire population
    fitness_scores = np.array([ga_fitness_function(ind) for ind in population])

    best_idx = np.argmax(fitness_scores)
    best_fitness_history.append(fitness_scores[best_idx])
    avg_fitness_history.append(fitness_scores.mean())

    print(f"  Generation {generation+1:02d}/{N_GENERATIONS} | "
          f"Best: {fitness_scores[best_idx]:.6f} | "
          f"Avg: {fitness_scores.mean():.6f} | "
          f"Features: {int(population[best_idx][:N_FEATURES].sum())}")

    # Elitism: preserve top performers
    elite_indices = np.argsort(fitness_scores)[::-1][:ELITE_SIZE]
    new_population = [population[i].copy() for i in elite_indices]

    # Create next generation
    while len(new_population) < POPULATION_SIZE:
        parent1 = tournament_selection(population, fitness_scores)
        parent2 = tournament_selection(population, fitness_scores)
        child1, child2 = single_point_crossover(parent1, parent2)
        new_population.append(bit_flip_mutation(child1))
        if len(new_population) < POPULATION_SIZE:
            new_population.append(bit_flip_mutation(child2))

    population = np.array(new_population)

ga_duration = (time.time() - ga_start_time) / 60

print(f"\nGA optimization completed in {ga_duration:.1f} minutes")

# Extract final optimal solution
final_fitness_scores = np.array([ga_fitness_function(ind) for ind in population])
optimal_index = np.argmax(final_fitness_scores)
optimal_chromosome = population[optimal_index]

# Extract final feature mask
optimal_feature_mask = optimal_chromosome[:N_FEATURES] == 1
optimal_n_features = int(optimal_feature_mask.sum())

# Extract final hyperparameters
optimal_n_estimators = 50 + int(''.join(map(str, optimal_chromosome[N_FEATURES:N_FEATURES+8].astype(int))), 2) % 151
optimal_max_depth = 5 + int(''.join(map(str, optimal_chromosome[N_FEATURES+8:N_FEATURES+16].astype(int))), 2) % 31
optimal_min_samples_split = 2 + int(''.join(map(str, optimal_chromosome[N_FEATURES+16:N_FEATURES+24].astype(int))), 2) % 19
optimal_min_samples_leaf = 1 + int(''.join(map(str, optimal_chromosome[N_FEATURES+24:N_FEATURES+32].astype(int))), 2) % 10

print(f"\n Final Optimal Solution (PSO-GA Hybrid):")
print(f"  Selected features: {optimal_n_features}/{N_FEATURES} ({optimal_n_features/N_FEATURES*100:.1f}%)")
print(f"  Hyperparameters:")
print(f"    n_estimators:      {optimal_n_estimators}")
print(f"    max_depth:         {optimal_max_depth}")
print(f"    min_samples_split: {optimal_min_samples_split}")
print(f"    min_samples_leaf:  {optimal_min_samples_leaf}")

# Compare with PSO
print(f"\n PSO vs GA Comparison:")
print(f"  PSO features: {pso_n_features}")
print(f"  GA features:  {optimal_n_features}")
print(f"  Improvement:  {pso_n_features - optimal_n_features} fewer features" if optimal_n_features < pso_n_features else "GA maintained same feature count")

PHASE 2: GENETIC ALGORITHM REFINEMENT

GA Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Parameter           | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Population size:    | 30
  Generations:        | 15
  Mutation rate:      | 0.05
  Elite size:         | 3
  Tournament size:    | 3
  Crossover rate:     | 0.9
  Seed:               | PSO optimal solution injected
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Estimated runtime: 1-2 hours

 Starting GA optimization...

  Generation 01/15 | Best: 0.990954 | Avg: 0.986160 | Features: 26
  Generation 02/15 | Best: 0.990954 | Avg: 0.987818 | Features: 26
  Generation 03/15 | Best: 0.990954 | Avg: 0.988513 | Features: 26
  Generation 04/15 | Best: 0.991367 | Avg: 0.989358 | Features: 25
  Generation 05/15 | Best: 0.991967 | Avg: 0.990098 | Features: 23
  Generation 06/15 | Best: 0.991967 | Avg: 0.990606 | Features: 23
  Generation 07/15 | Best: 0.992697 | Avg: 0.98997

In [11]:


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("=" * 60)
print("TRAINING FINAL MODEL ON FULL DATASET")
print("=" * 60)

print(f"""
Final Model Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Training samples: {X_train.shape[0]:,} (100% of data)
  Features:         {optimal_n_features} (selected by PSO-GA)
  Hyperparameters:
    - n_estimators:      {optimal_n_estimators}
    - max_depth:         {optimal_max_depth}
    - min_samples_split: {optimal_min_samples_split}
    - min_samples_leaf:  {optimal_min_samples_leaf}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

final_train_start = time.time()

final_model = RandomForestClassifier(
    n_estimators=optimal_n_estimators,
    max_depth=optimal_max_depth,
    min_samples_split=optimal_min_samples_split,
    min_samples_leaf=optimal_min_samples_leaf,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# Train on FULL dataset with selected features
final_model.fit(X_train[:, optimal_feature_mask], y_train)
final_training_duration = time.time() - final_train_start

print(f"Final model trained in {final_training_duration:.1f} seconds")

# Evaluate on test set
print("\n Evaluating on held-out test set...")
y_test_pred = final_model.predict(X_test[:, optimal_feature_mask])
y_test_proba = final_model.predict_proba(X_test[:, optimal_feature_mask])[:, 1]

# Calculate performance metrics
OPTIMIZED_PERFORMANCE = {
    "accuracy": accuracy_score(y_test, y_test_pred),
    "precision": precision_score(y_test, y_test_pred, average='weighted'),
    "recall": recall_score(y_test, y_test_pred, average='weighted'),
    "f1_score": f1_score(y_test, y_test_pred, average='weighted'),
    "roc_auc": roc_auc_score(y_test, y_test_proba),
    "n_features": optimal_n_features,
    "training_time_sec": final_training_duration
}

print("\n" + "=" * 60)
print("FINAL OPTIMIZED MODEL PERFORMANCE")
print("=" * 60)
print(f"""
  Metric         | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy:      {OPTIMIZED_PERFORMANCE['accuracy']:.4f}
  Precision:     {OPTIMIZED_PERFORMANCE['precision']:.4f}
  Recall:        {OPTIMIZED_PERFORMANCE['recall']:.4f}
  F1-Score:      {OPTIMIZED_PERFORMANCE['f1_score']:.4f}
  ROC-AUC:       {OPTIMIZED_PERFORMANCE['roc_auc']:.4f}
  Features used: {OPTIMIZED_PERFORMANCE['n_features']}
  Training time: {OPTIMIZED_PERFORMANCE['training_time_sec']:.1f} seconds
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

TRAINING FINAL MODEL ON FULL DATASET

Final Model Configuration:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Training samples: 1,765,652 (100% of data)
  Features:         18 (selected by PSO-GA)
  Hyperparameters:
    - n_estimators:      73
    - max_depth:         35
    - min_samples_split: 11
    - min_samples_leaf:  4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Final model trained in 483.9 seconds

 Evaluating on held-out test set...

FINAL OPTIMIZED MODEL PERFORMANCE

  Metric         | Value
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Accuracy:      0.9989
  Precision:     0.9989
  Recall:        0.9989
  F1-Score:      0.9989
  ROC-AUC:       1.0000
  Features used: 18
  Training time: 483.9 seconds
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [19]:

print("=" * 60)
print("DEFINING BASELINE PERFORMANCE")
print("=" * 60)

#  baseline results
BENCHMARK_PERFORMANCE = {
    "accuracy": 0.9977,
    "precision": 0.9977,
    "recall": 0.9977,
    "f1_score": 0.9977,
    "roc_auc": 0.9998,
    "n_features": 70,
    "training_time_sec": 144.5,
    "model": "Random Forest (Baseline)"
}

print("\n BENCHMARK_PERFORMANCE defined:")
print(f"   F1-Score: {BENCHMARK_PERFORMANCE['f1_score']:.4f}")
print(f"   Features: {BENCHMARK_PERFORMANCE['n_features']}")
print(f"   Accuracy: {BENCHMARK_PERFORMANCE['accuracy']:.4f}")


print(f"   F1-Score: {OPTIMIZED_PERFORMANCE['f1_score']:.4f}")
print(f"   Features: {OPTIMIZED_PERFORMANCE['n_features']}")

print("\n" + "=" * 60)
print("READY FOR COMPARISON CELL")
print("=" * 60)

DEFINING BASELINE PERFORMANCE

 BENCHMARK_PERFORMANCE defined:
   F1-Score: 0.9977
   Features: 70
   Accuracy: 0.9977
   F1-Score: 0.9989
   Features: 18

READY FOR COMPARISON CELL


In [26]:
print("=" * 60)
print("performance comparison")
print("=" * 60)

# calculate improvements
f1_improvement = OPTIMIZED_PERFORMANCE['f1_score'] - BENCHMARK_PERFORMANCE['f1_score']
f1_improvement_percent = (f1_improvement / BENCHMARK_PERFORMANCE['f1_score']) * 100
feature_reduction = BENCHMARK_PERFORMANCE['n_features'] - OPTIMIZED_PERFORMANCE['n_features']
feature_reduction_percent = (feature_reduction / BENCHMARK_PERFORMANCE['n_features']) * 100

print("\ncomparison table:")
print("=" * 80)
print(f"{'metric':<20} {'baseline':<25} {'pso-ga':<25} {'change':<15}")
print("-" * 80)
print(f"{'accuracy':<20} {BENCHMARK_PERFORMANCE['accuracy']:<25.4f} {OPTIMIZED_PERFORMANCE['accuracy']:<25.4f} {OPTIMIZED_PERFORMANCE['accuracy'] - BENCHMARK_PERFORMANCE['accuracy']:+15.4f}")
print(f"{'precision':<20} {BENCHMARK_PERFORMANCE['precision']:<25.4f} {OPTIMIZED_PERFORMANCE['precision']:<25.4f} {OPTIMIZED_PERFORMANCE['precision'] - BENCHMARK_PERFORMANCE['precision']:+15.4f}")
print(f"{'recall':<20} {BENCHMARK_PERFORMANCE['recall']:<25.4f} {OPTIMIZED_PERFORMANCE['recall']:<25.4f} {OPTIMIZED_PERFORMANCE['recall'] - BENCHMARK_PERFORMANCE['recall']:+15.4f}")
print(f"{'f1-score':<20} {BENCHMARK_PERFORMANCE['f1_score']:<25.4f} {OPTIMIZED_PERFORMANCE['f1_score']:<25.4f} {f1_improvement:+15.4f}")
print(f"{'roc-auc':<20} {BENCHMARK_PERFORMANCE['roc_auc']:<25.4f} {OPTIMIZED_PERFORMANCE['roc_auc']:<25.4f} {OPTIMIZED_PERFORMANCE['roc_auc'] - BENCHMARK_PERFORMANCE['roc_auc']:+15.4f}")
print(f"{'features':<20} {BENCHMARK_PERFORMANCE['n_features']:<25} {OPTIMIZED_PERFORMANCE['n_features']:<25} {feature_reduction:+15}")
print("-" * 80)

print("\n" + "=" * 60)
print("summary of improvements")
print("=" * 60)
print(f"""
  metric                           | improvement
----------------------------------------------------------------------
  f1-score improvement:            | +{f1_improvement:.4f} (+{f1_improvement_percent:.2f}%)
  feature reduction:               | {feature_reduction} features ({feature_reduction_percent:.1f}%)
  model complexity reduction:      | {BENCHMARK_PERFORMANCE['n_features']} to {OPTIMIZED_PERFORMANCE['n_features']} features
  optimization efficiency:         | used 5 percent of data for optimization
  final training data:             | 100 percent of data
----------------------------------------------------------------------
""")

# determine if optimization was successful
if OPTIMIZED_PERFORMANCE['f1_score'] > BENCHMARK_PERFORMANCE['f1_score']:
    print("result: optimization successful")
    print(f"  features used: {OPTIMIZED_PERFORMANCE['n_features']} vs {BENCHMARK_PERFORMANCE['n_features']}")
    print(f"  f1 improvement: +{f1_improvement:.4f}")
else:
    print("result: optimization did not exceed baseline.")


performance comparison

comparison table:
metric               baseline                  pso-ga                    change         
--------------------------------------------------------------------------------
accuracy             0.9977                    0.9989                            +0.0012
precision            0.9977                    0.9989                            +0.0012
recall               0.9977                    0.9989                            +0.0012
f1-score             0.9977                    0.9989                            +0.0012
roc-auc              0.9998                    1.0000                            +0.0002
features             70                        18                                    +52
--------------------------------------------------------------------------------

summary of improvements

  metric                           | improvement
----------------------------------------------------------------------
  f1-score improvement:    